# Week 33, Day 1: Type hints: the basics

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def area(width: float, height: float) -> float:
    return width * height

print(area(3, 4))
print(area("ab", 3))
print(area.__annotations__)

In [ ]:
from datetime import date

def add_vat(pence: int, rate: int = 20) -> int:
    return pence + pence * rate // 100

def greet(name: str, shout: bool = False) -> str:
    text = f"Hello, {name}"
    return text.upper() if shout else text

def days_until(day: date, today: date) -> int:
    return (day - today).days

def log(message: str) -> None:
    print(f"[log] {message}")

print(add_vat(1000), greet("Ada", shout=True), days_until(date(2026, 12, 25), date(2026, 11, 3)))
log("done")

In [ ]:
import inspect
import typing

def add_vat(pence: int, rate: int = 20) -> int:
    return pence + pence * rate // 100

print(add_vat.__annotations__)
print(typing.get_type_hints(add_vat))
sig = inspect.signature(add_vat)
print(sig)
for name, param in sig.parameters.items():
    print(name, param.annotation.__name__, param.default)

In [ ]:
import typing
from typing import Self

class Account:
    def __init__(self, owner: str, balance: int = 0) -> None:
        self.owner: str = owner
        self.balance: int = balance

    def deposit(self, amount: int) -> int:
        self.balance += amount
        return self.balance

    def can_afford(self, amount: int) -> bool:
        return amount <= self.balance

    def copy(self) -> Self:
        return type(self)(self.owner, self.balance)

    def transfer_to(self, other: "Account", amount: int) -> None:
        self.balance -= amount
        other.balance += amount

a, b = Account("Ada", 500), Account("Bo")
a.transfer_to(b, 200)
print(a.balance, b.balance, a.copy().owner, typing.get_type_hints(Account.transfer_to))

> **Key idea.** Type hints say what types values should have: `def f(x: int, y: str = "a") -> bool:`, `-> None` for no useful result, `total: int = 0` for variables, any class as a type. Python stores them (`__annotations__`, `typing.get_type_hints`, `inspect.signature`) but doesn't check them when running; editors and type checkers such as mypy use them to find bugs before the program runs. Annotate method parameters but not `self`; use `-> Self` for methods returning their own class, and quote a class name used inside its own body. Annotate every signature; let the checker infer most variables.

### Exercise: Annotating functions

Add type hints to the three functions, without changing what they do: `vat` takes and returns whole numbers of pence (the rate is a whole number too), `label` takes a name (text) and an optional flag saying whether to shout, and returns text, and `report` prints a message and returns nothing.

In [ ]:
def vat(pence, rate=20):
    return pence * rate // 100

def label(name, shout=False):
    return name.upper() if shout else name.title()

def report(message):
    print("REPORT:", message)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h\n    assert h(vat) == {\"pence\": int, \"rate\": int, \"return\": int}, f\"vat: {h(vat)}\"\n    assert h(label) == {\"name\": str, \"shout\": bool, \"return\": str}, f\"label: {h(label)}\"\n    assert h(report) == {\"message\": str, \"return\": type(None)}, f\"report: {h(report)}\"\n    assert vat(1000) == 200 and label(\"ada\") == \"Ada\" and label(\"ada\", True) == \"ADA\"")

### Exercise: Annotating a class

Add type hints to `Account`: the owner is text, amounts are whole numbers of pence, `__init__` returns nothing, `can_afford` gives True or False, and `copy` returns an object of the same class (use `Self`).

In [ ]:
from typing import Self

class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    def can_afford(self, amount):
        return amount <= self.balance

    def copy(self):
        return type(self)(self.owner, self.balance)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h, Self\n    A = ns[\"Account\"]\n    assert h(A.__init__) == {\"owner\": str, \"balance\": int, \"return\": type(None)}, f\"__init__: {h(A.__init__)}\"\n    assert h(A.deposit) == {\"amount\": int, \"return\": int} and h(A.can_afford) == {\"amount\": int, \"return\": bool}\n    assert h(A.copy) == {\"return\": Self}, f\"copy: {h(A.copy)}\"\n    a = A(\"Ada\", 100)\n    assert a.deposit(50) == 150 and a.can_afford(150) and a.copy().balance == 150")

### Exercise: Showing a signature

Write `describe(func)`, which builds a readable signature from a function's name, parameters and hints, like `vat(pence: int, rate: int = 20) -> int`. Use `inspect.signature` for the parameters and defaults (with `repr` for each default) and `typing.get_type_hints` for the hints, written with each type's `__name__`. A parameter without a hint is just its name; with no return hint, leave off the `->` part. Only simple class hints (like `int` or `str`) will be used.

In [ ]:
import inspect
import typing

def describe(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_describe():\n    \"signatures\"\n    def vat(pence: int, rate: int = 20) -> int: ...\n    def greet(name, greeting: str = \"Hello\"): ...\n    def nothing() -> None: ...\n    def plain(a, b=None): ...\n    assert describe(vat) == \"vat(pence: int, rate: int = 20) -> int\", f\"got {describe(vat)!r}\"\n    assert describe(greet) == \"greet(name, greeting: str = 'Hello')\", f\"got {describe(greet)!r}\"\n    assert describe(nothing) == \"nothing() -> NoneType\" and describe(plain) == \"plain(a, b=None)\"")

### Exercise: Checking arguments while running

Python doesn't check hints, but a program can. Write `call_checked(func, *args)`: for each positional argument that has a simple class hint, check `isinstance(arg, hint)`, raising `TypeError` with a message like `rate must be int, not str` for the first one that fails; then call the function and return its result. Arguments without hints aren't checked. (Remember that `bool` is a subclass of `int`, so `True` passes an `int` check; that's fine here.)

In [ ]:
import inspect
import typing

def call_checked(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checked():\n    \"checking\"\n    def vat(pence: int, rate: int = 20) -> int:\n        return pence * rate // 100\n    def greet(name: str, times) -> str:\n        return name * times\n    assert call_checked(vat, 1000) == 200 and call_checked(vat, 1000, 5) == 50 and call_checked(greet, \"ha\", 3) == \"hahaha\"\n    for f, args, msg in [(vat, (1000, \"5\"), \"rate must be int, not str\"), (vat, (9.5,), \"pence must be int, not float\"), (greet, (3, 3), \"name must be str, not int\")]:\n        try:\n            call_checked(f, *args)\n            assert False, f\"should raise TypeError: {msg}\"\n        except TypeError as err:\n            assert str(err) == msg, f\"message was {str(err)!r}\"")

### Bug hunt: Not really an int

mypy reports `error: Incompatible return value type (got "str", expected "int")  [return-value]` for `parse_age`, and the program crashes with `TypeError: can only concatenate str (not "int") to str`. Fix `parse_age` so it keeps its promise, and the program prints `32`.

In [ ]:
def parse_age(text: str) -> int:
    return text.strip()

age = parse_age(" 31 ")
print(age + 1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a real int\"\n    expect_output(\"32\")\n    assert parse_age(\"7\") == 7")

### Bug hunt: The wrong way round

mypy reports two `arg-type` errors on the last line: `Argument 1 to "transfer" has incompatible type "str"; expected "int"` and `Argument 2 to "transfer" has incompatible type "int"; expected "str"`. When run, it prints nonsense instead of `Sent 500 to Ada`. Fix the call.

In [ ]:
def transfer(amount: int, to: str) -> str:
    return f"Sent {amount} to {to}"

print(transfer("Ada", 500))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sent\"\n    expect_output(\"Sent 500 to Ada\")")

### Bug hunt: Nothing to multiply

mypy reports `"total" does not return a value (it only ever returns None)  [func-returns-value]`, and running it crashes with `TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'`. `total` should return the sum (and still print it). Fix the function, including its hint, so the program prints `350` then `70.0`.

In [ ]:
def total(prices: list[int]) -> None:
    print(sum(prices))

vat = total([100, 250]) * 0.2
print(vat)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the VAT\"\n    expect_output(\"350\\n70.0\")\n    import typing\n    assert typing.get_type_hints(total)[\"return\"] is int, \"total now returns an int\"")

### Puzzle: An annotated function

Arrange the lines into a fully annotated function that returns the price of `qty` tickets, with a group discount. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import typing

def f(a: int, b: str = "x") -> bool:
    return len(b) > a

def g(n: int) -> int:
    return str(n)

p = ___   # f(0)
q = ___   # g(5)
r = ___   # typing.get_type_hints(f)["return"].__name__
s = ___   # list(f.__annotations__)
t = ___   # f(True, "ab")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, \"5\", \"bool\"), \"p to r\"\n    assert s == [\"a\", \"b\", \"return\"] and t is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a fully typed basket

Write a small, fully annotated shopping module. Every function and method must have hints on all its parameters (except `self`) and its return value, as described.

- `Item`, a dataclass with `name: str`, `pence: int` and `qty: int = 1`, and `total(self) -> int` (pence times qty). `__post_init__` (annotated `-> None`) raises `ValueError` if `pence` is below 0 or `qty` below 1.
- `Basket`, a class with `__init__(self, owner: str) -> None`, and:
  - `add(self, item: Item) -> Self`, which adds the item and returns the basket itself, so calls can be chained: `b.add(x).add(y)`;
  - `remove(self, name: str) -> bool`, which removes the first item with that name and says whether it found one;
  - `total(self) -> int`, the total of all items;
  - `count(self) -> int`, the number of things (the sum of the quantities).
- `money(pence: int) -> str`, giving `£12.50` (and `-£3.05` for negative amounts).
- `receipt(basket: Basket) -> str`, giving one line per item like `Rice x2: £9.00`, then `Total: £12.50`, joined by newlines.

In [ ]:
from dataclasses import dataclass
from typing import Self

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"every hint\"\n    from typing import get_type_hints as h, Self\n    I, B = ns[\"Item\"], ns[\"Basket\"]\n    assert h(I) == {\"name\": str, \"pence\": int, \"qty\": int}, f\"Item fields: {h(I)}\"\n    assert h(I.total) == {\"return\": int} and h(I.__post_init__) == {\"return\": type(None)}\n    assert h(B.__init__) == {\"owner\": str, \"return\": type(None)} and h(B.add) == {\"item\": I, \"return\": Self}\n    assert h(B.remove) == {\"name\": str, \"return\": bool} and h(B.total) == {\"return\": int} and h(B.count) == {\"return\": int}\n    assert h(money) == {\"pence\": int, \"return\": str} and h(receipt) == {\"basket\": B, \"return\": str}\ndef test_behaviour():\n    \"the basket\"\n    I, B = ns[\"Item\"], ns[\"Basket\"]\n    b = B(\"Ada\")\n    assert b.add(I(\"Rice\", 450, 2)).add(I(\"Oil\", 350)) is b\n    assert b.total() == 1250 and b.count() == 3 and money(1250) == \"\u00a312.50\" and money(-305) == \"-\u00a33.05\" and money(7) == \"\u00a30.07\"\n    assert receipt(b) == \"Rice x2: \u00a39.00\\nOil x1: \u00a33.50\\nTotal: \u00a312.50\"\n    assert b.remove(\"Oil\") is True and b.remove(\"Oil\") is False and b.total() == 900\n    assert receipt(B(\"x\")) == \"Total: \u00a30.00\"\n    for bad in [lambda: I(\"x\", -1), lambda: I(\"x\", 5, 0)]:\n        try:\n            bad()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass")

## Warm-up

The hints say `area` takes two floats and returns a float, yet `area("ab", 3)` ran happily and returned a string. Python **stores** the hints (they're in `__annotations__`) but **never checks** them when the program runs. So what are they for? That's today's question. The short answer: other tools read them, and so do people.

## Writing hints

A **type hint** (or annotation) says what type a value should have. For a function:

```python
def add_vat(pence: int, rate: int = 20) -> int:
    return pence + pence * rate // 100
```

- After each parameter name: a colon, then the type. A default goes **after** the hint: `rate: int = 20` (with spaces round the `=`, unlike unhinted defaults).
- After the closing bracket: `->` and the type of the return value, then the colon.
- A function that returns nothing useful is written `-> None`.
- Any class can be a type: the built-ins (`int`, `float`, `str`, `bool`, `bytes`), your own classes, and those from the standard library (`datetime.date`, `Fraction`).

Variables can be annotated too, though it's rarely needed, because a type checker works out a variable's type from what's assigned to it:

```python
total: int = 0
name: str = input("Name: ")
```

Two conveniences, which type checkers allow on purpose: an `int` is accepted where a `float` is expected (every whole number is a sensible float), and a `bool` is accepted where an `int` is expected (Week 30: `bool` is a subclass of `int`).

## Why hints help

If Python doesn't check hints, who does?

- **People.** `def transfer(amount: int, to: str) -> str` tells a reader at a glance what goes in and comes out. That beats a docstring, because it can't silently drift out of date when tools check it.
- **Editors.** VS Code (Week 22) uses hints to suggest methods as you type (it knows `name` is a `str`, so `name.` offers `upper`, `strip`, ...) and to underline mistakes.
- **Type checkers.** Tools such as **mypy** read your whole program, follow every value through every function, and report places where the types can't fit, **without running anything**. Day 3 is about running mypy; for now, here's what it says about some buggy code.

```python
def parse_age(text: str) -> int:
    return text.strip()

def transfer(amount: int, to: str) -> str:
    return f"Sent {amount} to {to}"

transfer("Ada", 500)

def total(prices: list[int]) -> None:
    print(sum(prices))

vat = total([100, 250]) * 0.2
```

```text
bugs.py:2: error: Incompatible return value type (got "str", expected "int")  [return-value]
bugs.py:7: error: Argument 1 to "transfer" has incompatible type "str"; expected "int"  [arg-type]
bugs.py:7: error: Argument 2 to "transfer" has incompatible type "int"; expected "str"  [arg-type]
bugs.py:12: error: "total" does not return a value (it only ever returns None)  [func-returns-value]
bugs.py:12: error: Unsupported operand types for * ("None" and "float")  [operator]
Found 5 errors in 1 file (checked 1 source file)
```

Every one of those is a real bug. Two of them (`parse_age` and `transfer`) wouldn't even crash: they'd quietly produce wrong results, perhaps months later. A type checker finds them in a second, which is why most large Python projects use one. Tests (Week 24) and type checking complement each other: tests check that the code does the right thing for the cases you try; a type checker checks that every call fits, for every case.

## Hints are data

Python keeps a function's hints, and you can read them while the program runs. That's how editors, documentation tools, dataclasses and some libraries use them:

- `func.__annotations__` is a dictionary from parameter name (and `"return"`) to the hint.
- `typing.get_type_hints(func)` gives the same, and also works out hints written as strings (below). Prefer it.
- `inspect.signature(func)` describes the whole signature: names, defaults and hints. `help(func)` shows it too.

You'll use this in Day 5 to check types while a program runs, at the places where that's worth doing.

## Hints in classes

Methods are annotated like functions, without a hint on `self`. `__init__` returns `-> None`. Attributes are usually annotated where they're first set, in `__init__` (or as fields, in a dataclass):

- A method that returns an object of its own class is annotated `-> Self` (from `typing`). For a child class, `Self` means the child, which matches `type(self)(...)` from Week 30.
- Inside a class body, the class's own name isn't finished yet, so older Pythons can't use it in a hint directly. Write it as a **string**, `"Account"`: type checkers and `get_type_hints` read it as the class. (From Python 3.14 the quotes aren't needed, but they still work, and you'll see them in a lot of code.)

## What to annotate

A practical rule that most projects follow:

- **Annotate every function and method signature** you write: the parameters and the return value. That's where hints help most, and a type checker can't guess what callers will pass.
- **Don't annotate most local variables**: `total = 0` is obviously an `int`, and the checker infers it. Annotate a variable only when the checker can't tell, most often an empty collection (`names: list[str] = []`, tomorrow).
- **Be truthful.** A wrong hint is worse than none, because readers and tools believe it.
- **Hints don't replace checks.** If a function must reject bad input from outside your program (a form, a file), it still needs `if` and `raise`, because nothing checks hints when the program runs.

## Practice

## Quiz

## Daily challenge

## Before you go

Take three functions you wrote in earlier weeks and annotate them. Whenever you can't decide what a hint should be, notice why: it often means the function does too much. Tomorrow: hints for lists, dictionaries and tuples, and for values that might be `None`.